# BikeZ-ETH Data Visualization Tools Tutorial

This notebook walks through generating both visualization tools available for BikeZ-ETH trajectory data: a per-trajectory lane-coordinate debug map, and a fleet-wide animated map of all vehicles at a site/timeslot.


**Contents:**

1. Load trajectory data and site geometry for a given location/timeslot.
3. Generate a lane-coordinate debug map for a single trajectory.
4. Generate a fleet-wide animated trajectory map for both bicycles and vehicles at a given location/timeslot combination.

**Prerequisites:** see the main `README.md` for dataset setup (`BikeZ_Config.data_root`, `BikeZ_Config.subsampled_data_root`) and the `registry_location{loc_num}.pkl` files this notebook expects under `../data/`. Run this notebook from the `src/` directory so relative imports resolve correctly.

In [1]:
import pickle
import pandas as pd

In [2]:
from _constants import BikeZ_Config

BikeZ_Config = BikeZ_Config()
subsampled_data_root = BikeZ_Config.subsampled_data_root

date      = "2025-09-30"
mode      = "bike"
timeslot  = "PM1"
loc_num   = 13

intersection, code = BikeZ_Config.get_intersection_code(date, loc_num, timeslot)
print(f"Intersection: {intersection}, Code: {code}")
XY_2056_Bounds = BikeZ_Config.XY_2056_Bounds[date][(intersection, code)]
X_2056_offset = XY_2056_Bounds[0][0]
Y_2056_offset = XY_2056_Bounds[1][0]

Intersection: D2, Code: I


## Import Trajectory Data + Site Geometry Configuration

Trajectory data and the site's geometry registries are stored separately. The registry is always stored in `../data/` folder.

In [3]:
# Load Trajectories
filename = f"location_{loc_num}/{loc_num}_{mode}s_{date}_{timeslot}_lane.csv"
df = pd.read_csv(subsampled_data_root + filename)
df.columns

Index(['veh_id', 'veh_type', 'datetime', 'time', 'x_act_ekf', 'y_act_ekf',
       'x_ekf', 'y_ekf', 'lon_ekf', 'lat_ekf', 'speed_ekf', 'a_ekf',
       'angle_ekf', 'angular_vel_ekf', 'in_gap', 'off_grid', 'movement_key',
       'segment_id', 'segment_type', 'segment_role', 'match_quality',
       'is_fallback', 'is_reverse', 's_native', 'd_native', 's_dot', 'd_dot',
       's_ddot', 'd_ddot', 'in_bike_lane', 'd_to_bike_boundary'],
      dtype='object')

In [4]:
# Load geometry, segment, and movement registries
registry_path = f'../data/registry_location{loc_num}.pkl'
registry = pickle.load(open(registry_path, 'rb'))
geometry_store    = registry['geometry_store']
segment_registry  = registry['segment_registry']
movement_registry = registry['movement_registry']
max_chain_length  = registry['metadata'].get('max_chain_length', 3)

These dict registries act as the static layer defining the site geometry and infrastructure.
- **`geometry_store`**: one entry per physical road axis --> the B-spline fit to its centerline, arc length, stop/yield lines.
- **`segment_registry`**: one entry per *directed* travel segment (a lane or a turn), e.g. `FlurstrS_NB` or `turn_FlurstrS_NB_2_FlurstrN_NB`. Also, it stores bike lane information (e.g. boundaries and width)
- **`movement_registry`**: one entry per full movement through the intersection --> an ordered chain of segments a bike travels along (approach → turn → departure).

## Generate a lane coordinate debug map for a single bicycle or vehicle trajectory

Interactive single-file HTML visualisation for one bicycle trajectory after lane coordinate transformation. Combines a Leaflet satellite map with three linked Plotly panels, all driven by a shared playback animation.

In [5]:
# Step 1: Select a specific trajectory to visualize
bike_id = 8
bike_df = df[df['veh_id'] == bike_id].copy()
bike_df = bike_df.sort_values(by='time')

# Step 2: Compute travel-directed lane coordinates
from tools_lane_coords_V4 import compute_travel_directed_s_d

bike_df = compute_travel_directed_s_d(bike_df, segment_registry, geometry_store)
# Adds 's', 'd', 'cumulative_s' columns to bike_df
bike_df.columns

Index(['veh_id', 'veh_type', 'datetime', 'time', 'x_act_ekf', 'y_act_ekf',
       'x_ekf', 'y_ekf', 'lon_ekf', 'lat_ekf', 'speed_ekf', 'a_ekf',
       'angle_ekf', 'angular_vel_ekf', 'in_gap', 'off_grid', 'movement_key',
       'segment_id', 'segment_type', 'segment_role', 'match_quality',
       'is_fallback', 'is_reverse', 's_native', 'd_native', 's_dot', 'd_dot',
       's_ddot', 'd_ddot', 'in_bike_lane', 'd_to_bike_boundary', 's', 'd',
       'cumulative_s'],
      dtype='object')

In [6]:
from generate_debug_viz import generate_bikelane_debug_map

output_html_path =f'../maps/debug_map_location{loc_num}_{timeslot}_{mode}_{bike_id}.html'

generate_bikelane_debug_map(
    bike_df,                                # DataFrame after to_lane_coordinates(), single vehicle
    segment_registry,
    geometry_store,
    output_path=output_html_path,
)

# This opens the html in your browser automatically
import os
import webbrowser

full_path = os.path.abspath(output_html_path)
webbrowser.open(f'file://{full_path}')

Saved: ../maps/debug_map_location13_PM1_bike_8.html  (538 frames, 3 segments: BaslerstrE_WB → turn_BaslerstrE_WB_2_FlurstrS_SB → FlurstrS_NB)


True

**Left panel: Leaflet / swisstopo satellite map**

Togglable layer groups (via top-right layer control):
- Centerlines: Full spline for each matched segment; dashed for turns
- Validity polygons: Oriented corridor polygon per segment
- Change points: `s_change` marker on the centerline
- Bike lane bands: Inner boundary + outer edge + filled band
- Trajectory: GPS path coloured by `segment_id`

**Right panel: three linked Plotly charts**

| Plot | X-axis | Y-axis | Notes |
|---|---|---|---|
| A. Cumulative s vs d | Continuous s stitched across segment boundaries [m] | d [m] | vrect shading for `is_reverse` (salmon) and `in_bike_lane` (green) |
| B. s_native vs d_native | s_native [m] | d_native [m] | One trace per segment; time is the animation dimension |
| C. Speed, $\dot{s}$ , $\dot{d}$ vs time | t [s] | km/h | `speed_ekf` grey dashed; `s_dot` / `d_dot` solid/dotted per segment colour; vrect flags |

Click any plot to jump the scrubber to that position. All panels share the same segment colour palette.

## Generate Fleet-wide Animated Trajectory Map

Command-line script that renders all bicycle and vehicle trajectories at a given site and timeslot as an animated Leaflet map using TimestampedGeoJson. Bikes appear as small blue circles; vehicles as slightly larger red circles.

In [7]:
history_length = int(1) # in seconds

# This runs the command-line script to generate the timestamped map for all bicycles+vehicles trajectories in the dataset
!python generate_timestamped_map.py $date $intersection $code $timeslot True $history_length

Output is saved to `../maps/timestamped_trajectories_ALL_map_<date>_<intersection>_<timeslot>_<code>_history<history_len>s.html`. The next cell opens this visualization automatically.

In [8]:
output_html_path = f'../maps/timestamped_trajectories_ALL_map_{date}_{intersection}_{timeslot}_{code}_history{history_length}s.html'
full_path = os.path.abspath(output_html_path)
webbrowser.open(f'file://{full_path}')

True